In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/extracted-josn/latex_extracted.json


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## T5 Summarizer


In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
import json
import pandas as pd
from pandas import json_normalize
import torch

In [2]:
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True,
                                              use_safetensors=True,local_files_only=False,)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [60]:
def t5_generate(text):
    inputs = tokenizer(text,return_tensors="pt",
              truncation=True,max_length=512).to(device)

    out_ids = model.generate(inputs["input_ids"],
                  num_beams=4,max_length=350,
                            do_sample=False)

    summary = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return summary

In [61]:
def clean_summary(text):
    if not text:
        return ""

    text = re.sub(r"\$[^$]*\$", "", text)
    text = re.sub(r"\$\$[^$]*\$\$", "", text)
    text = re.sub(r"[#|&*{}_^\\\/]+", " ", text)
    text = re.sub(r"\\[a-zA-Z]+\*?", "", text)
    text = re.sub(r"\|[^|]*\|", " ", text)
    text = re.sub(r"(?i)section\s*\d+:?", "", text)
    text = re.sub(r"(?i)summary for section\s*\d+:?", "", text)
    text = re.sub(r"[-=]{2,}", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

In [62]:
def build_section_chunks(paper):
    chunks = []
    for sec in paper["sections"]:
        chunks.append({
            "title": paper["title"],
            "abstract": paper["abstract"],
            "section_name": sec["name"],
            "section_content": sec["content"]
        })
    return chunks

In [63]:
def format_equation_for_summary(eq):
    eq = eq.replace("\\\\", "\\\\\n")
    return f"```latex\n{eq}\n```"

In [64]:
def pick_top_equations_raw(paper, top_k=5):
    eqs = paper.get("equations", [])
    return eqs[:top_k]

In [65]:
def split_into_chunks(text, max_tokens=300):
    sentences = text.split(". ")
    chunks = []
    current = ""

    for s in sentences:
        if len(current.split()) + len(s.split()) < max_tokens:
            current += s + ". "
        else:
            chunks.append(current.strip())
            current = s + ". "
    if current:
        chunks.append(current.strip())
    return chunks

In [66]:
def summarize_abstract(paper):
    prompt = f"""
        ou are summarizing ONLY the abstract of a scientific research paper.

        Write a detailed academic summary that is 5–7 sentences long.
        Your summary MUST include:
        - the main problem or object studied,
        - the motivation or context (if implied),
        - the methods or approach used (e.g., computation, ML, geometry),
        - the key results or theorems,
        - the significance of those results.

        Strict rules:
        - Summarize ONLY the abstract, NOT any section.
        - Use clear academic writing, not bullet points.
        - DO NOT include equations or LaTeX.
        - DO NOT add new claims beyond what is described.
        - DO NOT shorten the meaning; expand it into full sentences.
        - DO NOT copy the abstract wording; rewrite in new sentences.

        Abstract Text:
        {paper['abstract']}

        ---
        Write the abstract summary below (5–7 full sentences):
            """
    return t5_generate(prompt)

In [67]:
def summarize_section(section, top_equations):
    content = section["section_content"]

    # STEP 1: split section into small chunks
    small_chunks = split_into_chunks(content, max_tokens=250)

    partial_summaries = []

    # STEP 2: summarize each small chunk
    for chunk_text in small_chunks:
        prompt = build_summary_prompt(
            {"section_name": section["section_name"], "section_content": chunk_text},
            top_equations
        )
        summary = t5_generate(prompt)
        partial_summaries.append(clean_summary(summary))

    # STEP 3: combine summaries into one
    combined = " ".join(partial_summaries)

    # STEP 4: compress final summary
    final_prompt = f"""
        Section: {section['section_name']}
        
        Combine and compress the following partial summaries into ONE clear section summary:
        
        {combined}
        
        Output ONLY the final summary (3–5 sentences).
            """

    final_summary = t5_generate(final_prompt)
    return clean_summary(final_summary)


In [68]:
def get_section_instruction(section_name):
    name = section_name.lower()

    if "introduction" in name:
        return """
        Summarize the INTRODUCTION by focusing on:
        - the motivation for the research,
        - what problem the authors want to solve,
        - why the problem is important,
        - the general idea of their approach.
        Do NOT include detailed results or experiments.
        """

    if "result" in name:
        return """
        Summarize the RESULTS section by focusing on:
        - the main theorems or mathematical claims,
        - the surjectivity criterion,
        - how the indeterminacy locus I_f determines surjectivity.
        Do NOT talk about motivation or experiments.
        """

    if "experiment" in name or "example" in name:
        return """
        Summarize the EXPERIMENTS/EXAMPLES section by focusing on:
        - the computational or ML methods used,
        - how Python or algorithms generated examples,
        - new explicit maps the authors constructed,
        - empirical observations that support the theory.
        Do NOT describe theorems.
        """

    return "Summarize this section accurately and clearly."

In [69]:
def build_summary_prompt(chunk, top_equations):
    eq_text = "\n\n".join(format_equation_for_summary(eq) for eq in top_equations)
    section_instruction = get_section_instruction(chunk['section_name'])

    prompt = f"""
    You are summarizing a section of a scientific mathematics paper.
    Follow the instructions below to summarize correctly.

    {section_instruction}

    Section Name: {chunk['section_name']}
    Section Content:
    {chunk['section_content']}

    Important Equations:
    {eq_text}

    Write a detailed, clear, human-friendly summary (5–7 sentences).
    """
    return prompt

In [70]:
def explain_equation_t5(eq_latex):
    prompt = (
        "The following is a LaTeX mathematical equation:\n\n"
        "```latex\n"
        f"{eq_latex}\n"
        "```\n\n"
        "Explain the equation clearly:\n"
        "1. Meaning of variables.\n"
        "2. Mathematical interpretation.\n"
        "3. Step-by-step derivation (as far as possible).\n"
        "4. Why this equation appears in the paper.\n"
        "5. What mathematical object/structure it represents.\n"
    )
    return t5_generate(prompt)

In [71]:
def build_final_report_json( abstract_summary, section_chunks,
        section_summaries, top_equations, derivations, paper):


    #  sections
    sections = []
    for chunk, summary in zip(section_chunks, section_summaries):
        sections.append({
            "section_name": chunk["section_name"],
            "section_summary": summary.strip()
        })

    #  important equations (formatted)
    important_equations = []
    for i, eq in enumerate(top_equations, 1):
        important_equations.append({
            "equation_no": f"Equation {i}",
            "equation": eq
        })

    #  equation explanations (formatted)
    equation_explanations = []
    for i, expl in enumerate(derivations, 1):
        equation_explanations.append({
            "equation_no": f"Equation {i}",
            "explanation": expl.strip()
        })

    #  final json object
    final_json = {
        "title": paper["title"],
        "authors": paper["authors"],
        "abstract": abstract_summary.strip(),
        "sections": sections,
        "important_equations": important_equations,
        "equation_explanations": equation_explanations
    }

    return final_json



In [72]:
json_path = "/kaggle/input/extracted-josn/latex_extracted.json"
with open(json_path, "r") as f:
    data = json.load(f)
paper = data[0]

In [73]:
section_chunks = build_section_chunks(paper)

In [74]:
top_equations = pick_top_equations_raw(paper, top_k=5)

In [75]:
abstract_summary = summarize_abstract(paper)

In [76]:
abstract_summary_clean = clean_summary(abstract_summary)
abstract_summary_clean

'We develop an experimental approach, based on some Python programming and Machine Learning, towards the classification of such maps; a couple of new explicit is constructed in this way. We also prove (via pure projective geometry) that a general non-regular cubic endomorphism of is surjective if and only if the set has cardinality at least .'

In [77]:
section_summaries = []
for section in section_chunks:
    print("Processing:", section['section_name'])
    summary = summarize_section(section, top_equations)
    print(summary)
    section_summaries.append(summary)

Processing: Introduction
This paper introduces a new theory of surjective maps in families, aiming to show that for certain rational maps the surjectivity is a . For every given cardinality , the set of all defining surjective maps is (including the set ) We extend the main result of by showing that all such surjective , with fixed generic , form an open set (see Section below) and develop an experimental approach towards a complete description of (cubic) surjective rational maps, which also yields new examples of them (see Section). The general idea of the authors' approach is that (surjective) cubic maps can be defined as a vector bundle together with an inclusion , where is a properly defined pull-back (see ). Thus one may regard every such as a certain connection on the (Schwarzenberger-like) bundle (cf. [ 2.2.2]Dol-CAG). In view of the present discussion, it is tempting to look for (hidden) symmetries associated with , which one may construct in terms of sublattices in for the fib

In [78]:
section_summaries

["This paper introduces a new theory of surjective maps in families, aiming to show that for certain rational maps the surjectivity is a . For every given cardinality , the set of all defining surjective maps is (including the set ) We extend the main result of by showing that all such surjective , with fixed generic , form an open set (see Section below) and develop an experimental approach towards a complete description of (cubic) surjective rational maps, which also yields new examples of them (see Section). The general idea of the authors' approach is that (surjective) cubic maps can be defined as a vector bundle together with an inclusion , where is a properly defined pull-back (see ). Thus one may regard every such as a certain connection on the (Schwarzenberger-like) bundle (cf. [ 2.2.2]Dol-CAG). In view of the present discussion, it is tempting to look for (hidden) symmetries associated with , which one may construct in terms of sublattices in for the fibers (cf. [ 2.2.2]Dol-CA

In [79]:
derivations = [explain_equation_t5(eq) for eq in top_equations]

In [80]:
final_data = build_final_report_json(
    abstract_summary,
    section_chunks,
    section_summaries,
    top_equations,
    derivations,
    paper
)
print(final_data)

{'title': 'Computations and ML for surjective rational maps', 'authors': ['Ilya Karzhemanov'], 'abstract': 'We develop an experimental approach, based on some Python programming and Machine Learning, towards the classification of such maps; a couple of new explicit $f$ is constructed in this way. We also prove (via pure projective geometry) that a general non-regular cubic endomorphism $f$ of $2$ is surjective if and only if the set $I_f$ has cardinality at least $3$.', 'sections': [{'section_name': 'Introduction', 'section_summary': "This paper introduces a new theory of surjective maps in families, aiming to show that for certain rational maps the surjectivity is a . For every given cardinality , the set of all defining surjective maps is (including the set ) We extend the main result of by showing that all such surjective , with fixed generic , form an open set (see Section below) and develop an experimental approach towards a complete description of (cubic) surjective rational maps

In [81]:
df = pd.json_normalize(final_data)
df

,title,authors,abstract,sections,important_equations,equation_explanations
0,Computations and ML for surjective rational maps,[Ilya Karzhemanov],"We develop an experimental approach, based on ...","[{'section_name': 'Introduction', 'section_sum...","[{'equation_no': 'Equation 1', 'equation': '\l...","[{'equation_no': 'Equation 1', 'explanation': ..."


In [83]:
df.to_csv("/kaggle/working/structured_summary.csv", index=False)